# 12. 검출 행에서 키워드군(버섯·고기·수산물·냉동/냉장)이 차지하는 비율

- 대상 텍스트: `prduct_nm` (제품명). 대소문자 무시
- 한 행이 여러 키워드군에 동시에 걸릴 수 있다 (예: 냉동 새우 → 수산물 + 냉동/냉장) → 군별 비율의 합은 100% 가 아니다. 겹침은 10-3 에서 따로 본다
- 한 글자 키워드(게·회·굴·육 등)는 다른 단어에 섞여 오탐이 많아 넣지 않았다. **키워드 목록은 아래 셀에서 고치면 된다**
- 키워드별 실제 걸린 제품명 예시를 12-4 에서 출력 → 오탐 확인용

In [1]:
import re
import pandas as pd

pd.set_option("display.width", 200)
pd.set_option("display.max_rows", 300)
pd.set_option("display.max_colwidth", 60)

df = pd.read_parquet(r"C:\cheon\cheon_wokespace\listeria2026\리스테리아ML\data\listeria_body.parquet")
df["pos"] = (df["judge_result"] == "검출").astype(int)
df["is_mush"] = df["prduct_nm"].astype(str).str.contains("버섯|MUSHROOM|Flammulina|ENOKI|팽이", case=False, regex=True)
print(f"전체 {len(df):,}행 / 검출 {df['pos'].sum():,}건 / 검출률 {df['pos'].mean() * 100:.3f}%")

전체 676,537행 / 검출 1,969건 / 검출률 0.291%


## 12-1. 키워드 정의

In [2]:
KW = {
    "버섯": ["버섯", "MUSHROOM", "Flammulina", "ENOKI", "팽이"],
    "고기": ["고기", "소고기", "쇠고기", "돼지", "한우", "한돈", "돈육", "우육", "계육", "정육", "양념육", "분쇄육", "육포",
             "닭", "치킨", "오리", "햄", "소시지", "소세지", "베이컨", "갈비", "삼겹", "목살", "등심", "안심", "불고기", "돈까스", "돈가스",
             "편육", "족발", "순대", "미트", "BEEF", "PORK", "CHICKEN", "MEAT", "HAM", "SAUSAGE", "BACON", "SALAMI", "PROSCIUTTO"],
    "수산물": ["수산", "생선", "어묵", "연어", "참치", "고등어", "새우", "오징어", "문어", "낙지", "쭈꾸미", "주꾸미", "꽃게", "대게", "게살",
               "조개", "홍합", "바지락", "전복", "굴", "명란", "창란", "젓갈", "멸치", "대구", "명태", "황태", "코다리", "장어", "광어", "우럭",
               "가자미", "삼치", "갈치", "꽁치", "미역", "다시마", "김밥김", "해삼", "멍게", "성게", "어란", "훈제연어",
               "SALMON", "TUNA", "SHRIMP", "PRAWN", "FISH", "SEAFOOD", "CRAB", "SQUID", "OCTOPUS", "COD", "ROE", "MUSSEL", "CLAM", "OYSTER", "EEL"],
    "냉동/냉장": ["냉동", "냉장", "급랭", "급냉", "동결", "FROZEN", "CHILLED", "REFRIGERAT", "FREEZ"],
}
pn = df["prduct_nm"].astype(str)
for k, words in KW.items():
    pat = "|".join(re.escape(w) for w in words)
    df[f"kw_{k}"] = pn.str.contains(pat, case=False, regex=True)
kw_cols = [f"kw_{k}" for k in KW]
df["kw_none"] = ~df[kw_cols].any(axis=1)
for k, words in KW.items():
    print(f"{k} ({len(words)}개): {', '.join(words)}")

버섯 (5개): 버섯, MUSHROOM, Flammulina, ENOKI, 팽이
고기 (41개): 고기, 소고기, 쇠고기, 돼지, 한우, 한돈, 돈육, 우육, 계육, 정육, 양념육, 분쇄육, 육포, 닭, 치킨, 오리, 햄, 소시지, 소세지, 베이컨, 갈비, 삼겹, 목살, 등심, 안심, 불고기, 돈까스, 돈가스, 편육, 족발, 순대, 미트, BEEF, PORK, CHICKEN, MEAT, HAM, SAUSAGE, BACON, SALAMI, PROSCIUTTO
수산물 (58개): 수산, 생선, 어묵, 연어, 참치, 고등어, 새우, 오징어, 문어, 낙지, 쭈꾸미, 주꾸미, 꽃게, 대게, 게살, 조개, 홍합, 바지락, 전복, 굴, 명란, 창란, 젓갈, 멸치, 대구, 명태, 황태, 코다리, 장어, 광어, 우럭, 가자미, 삼치, 갈치, 꽁치, 미역, 다시마, 김밥김, 해삼, 멍게, 성게, 어란, 훈제연어, SALMON, TUNA, SHRIMP, PRAWN, FISH, SEAFOOD, CRAB, SQUID, OCTOPUS, COD, ROE, MUSSEL, CLAM, OYSTER, EEL
냉동/냉장 (9개): 냉동, 냉장, 급랭, 급냉, 동결, FROZEN, CHILLED, REFRIGERAT, FREEZ


## 12-2. 검출 행 중 키워드군별 비율 (+ 전체 행 대비 비교, 키워드군별 검출률)

In [3]:
pos = df[df["pos"] == 1]
rows = []
for k in list(KW) + ["none"]:
    c = f"kw_{k}"
    rows.append({"키워드군": "어디에도 안 걸림" if k == "none" else k,
                 "검출행": int(pos[c].sum()), "검출행중비율(%)": round(pos[c].mean() * 100, 2),
                 "전체행": int(df[c].sum()), "전체행중비율(%)": round(df[c].mean() * 100, 2),
                 "키워드군 검출률(%)": round(df.loc[df[c], "pos"].mean() * 100, 3) if df[c].any() else None})
kw = pd.DataFrame(rows).set_index("키워드군")
kw["검출비중/행비중(배)"] = (kw["검출행중비율(%)"] / kw["전체행중비율(%)"]).round(2)
print(f"검출 {len(pos):,}건 기준 (한 행이 여러 군에 걸리면 중복 집계 → 합이 100% 아님) / 전체 평균 검출률 {df['pos'].mean() * 100:.3f}%")
display(kw)

검출 1,969건 기준 (한 행이 여러 군에 걸리면 중복 집계 → 합이 100% 아님) / 전체 평균 검출률 0.291%


,검출행,검출행중비율(%),전체행,전체행중비율(%),키워드군 검출률(%),검출비중/행비중(배)
키워드군,,,,,,
버섯,526,26.71,9851,1.46,5.340,18.29
고기,564,28.64,120759,17.85,0.467,1.60
수산물,245,12.44,38742,5.73,0.632,2.17
냉동/냉장,232,11.78,34767,5.14,0.667,2.29
어디에도 안 걸림,533,27.07,487159,72.01,0.109,0.38


## 12-3. 겹침 — 검출 행의 키워드군 조합별 건수 (배타적, 합 = 전체 검출)

In [4]:
combo = pos[kw_cols].apply(lambda r: " + ".join(k for k, v in zip(KW, r) if v) or "어디에도 안 걸림", axis=1)
cb = combo.value_counts().rename("검출건수").to_frame()
cb["비율(%)"] = (cb["검출건수"] / len(pos) * 100).round(2)
print(f"합계 {int(cb['검출건수'].sum()):,}건")
display(cb)

합계 1,969건


,검출건수,비율(%)
어디에도 안 걸림,533,27.07
버섯,519,26.36
고기,466,23.67
수산물,209,10.61
냉동/냉장,113,5.74
고기 + 냉동/냉장,93,4.72
수산물 + 냉동/냉장,24,1.22
버섯 + 수산물,7,0.36
고기 + 수산물,3,0.15
고기 + 수산물 + 냉동/냉장,2,0.10


## 12-3-1. 여러 키워드군에 동시에 걸린 검출 행 — 전부, 어떤 단어에 걸렸는지

- `걸린군` = 걸린 키워드군 조합, `걸린단어` = 군별로 실제 매칭된 키워드

In [5]:
def matched_words(name):
    s = str(name).upper()
    out = []
    for k, words in KW.items():
        hit = [w for w in words if w.upper() in s]
        if hit:
            out.append(f"{k}[{', '.join(hit)}]")
    return " / ".join(out)


multi = pos[pos[kw_cols].sum(axis=1) >= 2].copy()
multi["걸린군"] = combo.loc[multi.index]
multi["걸린단어"] = multi["prduct_nm"].map(matched_words)
print(f"여러 군에 걸린 검출 {len(multi):,}건 (검출 {len(pos):,}건 중 {len(multi) / len(pos) * 100:.2f}%)")
display(multi["걸린군"].value_counts().rename("검출건수").to_frame())

with pd.option_context("display.max_rows", None, "display.max_colwidth", 80):
    display(multi.sort_values(["걸린군", "prduct_nm"])[["걸린군", "prduct_nm", "걸린단어", "train_prdlst_nm"]].reset_index(drop=True))

여러 군에 걸린 검출 129건 (검출 1,969건 중 6.55%)


,검출건수
걸린군,
고기 + 냉동/냉장,93
수산물 + 냉동/냉장,24
버섯 + 수산물,7
고기 + 수산물,3
고기 + 수산물 + 냉동/냉장,2


,걸린군,prduct_nm,걸린단어,train_prdlst_nm
0,고기 + 냉동/냉장,경기 2공장_돈육(냉동),고기[돈육] / 냉동/냉장[냉동],"정액,수정란,종란"
1,고기 + 냉동/냉장,경기2 돈육 냉동,고기[돈육] / 냉동/냉장[냉동],"정액,수정란,종란"
2,고기 + 냉동/냉장,경기2 돈육(냉동),고기[돈육] / 냉동/냉장[냉동],식육가공품 및 포장육
3,고기 + 냉동/냉장,경기2 우육 냉동,고기[우육] / 냉동/냉장[냉동],"정액,수정란,종란"
4,고기 + 냉동/냉장,경기2 우육 냉장,고기[우육] / 냉동/냉장[냉장],"정액,수정란,종란"
5,고기 + 냉동/냉장,경기2 우육(냉장),고기[우육] / 냉동/냉장[냉장],식육가공품 및 포장육
6,고기 + 냉동/냉장,계품닭(120g)급냉전,고기[닭] / 냉동/냉장[급냉],기타기준규격외
7,고기 + 냉동/냉장,고기완자-냉장전,고기[고기] / 냉동/냉장[냉장],기타기준규격외
8,고기 + 냉동/냉장,고기완자-냉장후,고기[고기] / 냉동/냉장[냉장],기타기준규격외
9,고기 + 냉동/냉장,국산 냉동 삼겹살(금천 대전),고기[삼겹] / 냉동/냉장[냉동],"정액,수정란,종란"


## 12-4. 키워드군별로 실제 걸린 검출 제품명 상위 15개 — 오탐 확인용

In [6]:
for k in KW:
    s = pos.loc[pos[f"kw_{k}"], "prduct_nm"].astype(str).value_counts()
    print(f"===== {k}: 검출 {int(pos[f'kw_{k}'].sum())}건 / 제품명 {len(s)}종")
    display(s.head(15).rename("검출건수").to_frame())

===== 버섯: 검출 526건 / 제품명 375종


,검출건수
prduct_nm,
팽이버섯,9
ENOKI MUSHROOM GR5-3,6
수출용 팽이버섯 (2번),6
ENOKI MUSHROOM GR4-1,6
수출용 팽이버섯 (3번),6
ENOKI MUSHROOM GR4-2,6
ENOKI MUSHROOM GR1-3,5
수출용 팽이버섯 (7번),5
ENOKI MUSHROOM GR4-3,5


===== 고기: 검출 564건 / 제품명 484종


,검출건수
prduct_nm,
냉동쇠고기,9
냉동돼지고기,7
닭고기염지육,7
한우목심(냉장),5
춘천닭갈비,4
요니네 한우 한판,4
한우 목심(냉장),4
햄,4
가마로순살강정육 1차,3


===== 수산물: 검출 245건 / 제품명 189종


,검출건수
prduct_nm,
광어회,10
연어회,10
연어필렛,6
홍새우회,5
냉장 연어 필렛,4
연어,4
연어(캐나다산),4
냉장연어살,4
생연어회,3


===== 냉동/냉장: 검출 232건 / 제품명 182종


,검출건수
prduct_nm,
냉동쇠고기,9
냉동돼지고기,7
한우목심(냉장),5
냉동다랑어(눈다랑어 필렛횟감),5
냉장 연어 필렛,4
냉동황새치 필렛횟감,4
한우 목심(냉장),4
"냉동새치(황새치,F,횟감)",4
냉장연어살,4


## 12-5. 어디에도 안 걸린 검출 행 — 품목(`train_prdlst_nm`)별 건수, 제품명별 건수 전부, 행 전체 목록

In [7]:
nn = pos[pos["kw_none"]]
print(f"어디에도 안 걸린 검출 {len(nn):,}건")
with pd.option_context("display.max_rows", None, "display.max_colwidth", 80):
    print("[1] 품목별 검출 건수")
    display(nn["train_prdlst_nm"].astype(str).value_counts().rename("검출건수").to_frame())
    s = nn["prduct_nm"].astype(str).value_counts()
    print(f"[2] 제품명별 검출 건수 — 전부 ({len(s):,}종)")
    display(s.rename("검출건수").to_frame())
    print(f"[3] 행 전체 목록 ({len(nn):,}행, 품목·제품명 순)")
    display(nn.sort_values(["train_prdlst_nm", "prduct_nm"])[["train_prdlst_nm", "prduct_nm", "mnfctur_cmpny_nm", "mnfctur_de"]].reset_index(drop=True))

어디에도 안 걸린 검출 533건
[1] 품목별 검출 건수


,검출건수
train_prdlst_nm,
기타기준규격외,293
식육가공품 및 포장육,87
"정액,수정란,종란",77
유가공품류,40
즉석식품류,23
축산물,3
기타식품류,3
기타식물류,2
음료류,1


[2] 제품명별 검출 건수 — 전부 (484종)


,검출건수
prduct_nm,
포장육 완제품,5
연성치즈,5
완제품1,3
클린콜,3
초밥세트 24P,3
고르곤졸라 돌체,3
얇은고명지단,2
완제품2,2
자연치즈,2


[3] 행 전체 목록 (533행, 품목·제품명 순)


,train_prdlst_nm,prduct_nm,mnfctur_cmpny_nm,mnfctur_de
0,견과 종실류,밤(탈피),<NA>,2022-11-10
1,"과자류, 빵류 또는 떡류",고향만두교자,<NA>,2017-02-09
2,기타기준규격외,(포장육)원료 입고전1,<NA>,2026-06-16
3,기타기준규격외,1-10,<NA>,2021-12-06
4,기타기준규격외,1-10,<NA>,2021-12-20
5,기타기준규격외,1-13,<NA>,2021-09-27
6,기타기준규격외,1-14,<NA>,2021-09-13
7,기타기준규격외,1-15,<NA>,2021-12-06
8,기타기준규격외,1-15,<NA>,2021-04-19
9,기타기준규격외,1-5,<NA>,2021-03-08


## 정리 (셀 출력 수치만)

- 검출 1,969건 중 (중복 허용): 버섯 26.71%, 고기 28.64%, 수산물 12.44%, 냉동/냉장 11.78%, 어디에도 안 걸림 27.07%
- 행 비중 대비 검출 비중: 버섯 18.29배, 냉동/냉장 2.29배, 수산물 2.17배, 고기 1.60배. 키워드군 검출률은 버섯 5.340%, 냉동/냉장 0.667%, 수산물 0.632%, 고기 0.467% (전체 0.291%)
- 겹침: 고기+냉동/냉장 93건, 수산물+냉동/냉장 24건. 냉동/냉장 단독은 113건
- 수산물 검출 상위는 광어회·연어회(각 10건), 연어 필렛 등 → 날것으로 먹는 횟감류
- **키워드 누락 확인됨**: 어디에도 안 걸린 533건에 `육회`, `육사시미`, `원료육`, `목심`, `뒷다리살`, `우둔`, `하몽`, `잠봉`, `포장육 완제품` 등 고기류와 `연성치즈`, `고르곤졸라`, `탈레지오` 등 치즈류가 있음 → 고기 비율은 과소 추정. 품목도 기타기준규격외 293, 식육가공품 및 포장육 87, 정액,수정란,종란 77, 유가공품류 40